### churn modeling pdf data gold layer

### benchmark overview

In [0]:
%sql
CREATE OR REPLACE TABLE customer_360.gold.churn_benchmark_overview AS

WITH base AS (
  SELECT * FROM customer_360.silver.silver_churn_benchmark
),


stats AS (
  SELECT
    AVG(credit_score)                                    AS avg_credit_score,
    PERCENTILE(credit_score, 0.5)                        AS median_credit_score,
    AVG(age)                                             AS avg_age,
    PERCENTILE(age, 0.5)                                 AS median_age,
    AVG(balance)                                          AS avg_balance,
    PERCENTILE(balance, 0.5)                              AS median_balance,
    AVG(estimated_salary)                                 AS avg_salary,
    AVG(CASE WHEN is_churned THEN 1.0 ELSE 0 END)         AS overall_churn_rate,
    COUNT(*)                                              AS total_customers
  FROM base
)

SELECT
  b.geography,
  b.gender,
  b.age_band,
  b.credit_score_band,
  b.balance_tier,
  b.product_category,
  b.salary_band,
  b.churn_label,

  COUNT(*)                                                AS customers,
  SUM(CASE WHEN b.is_churned THEN 1 ELSE 0 END)           AS churned_customers,
  ROUND(AVG(CASE WHEN b.is_churned THEN 1.0 ELSE 0 END) * 100, 2) AS churn_rate_pct,

  ROUND(AVG(b.credit_score), 1)                           AS avg_credit_score,
  ROUND(AVG(b.age), 1)                                    AS avg_age,
  ROUND(AVG(b.balance), 2)                                AS avg_balance,
  ROUND(AVG(b.estimated_salary), 2)                       AS avg_salary,
  ROUND(AVG(b.num_of_products), 2)                        AS avg_products,
  ROUND(AVG(CASE WHEN b.has_credit_card THEN 1.0 ELSE 0 END) * 100, 1) AS pct_with_card,
  ROUND(AVG(CASE WHEN b.is_active THEN 1.0 ELSE 0 END) * 100, 1)       AS pct_active,

 
  ROUND(
    (AVG(CASE WHEN b.is_churned THEN 1.0 ELSE 0 END) * 100)
    - (SELECT overall_churn_rate * 100 FROM stats)
  , 2)                                                     AS churn_rate_vs_overall_diff,

  (SELECT ROUND(overall_churn_rate * 100, 2) FROM stats)  AS overall_churn_rate_pct,
  (SELECT total_customers FROM stats)                      AS total_dataset_customers,

  current_timestamp()                                      AS gold_processed_time

FROM base b
GROUP BY
  b.geography, b.gender, b.age_band, b.credit_score_band,
  b.balance_tier, b.product_category, b.salary_band, b.churn_label;

### benchmark by dimension

In [0]:
%sql

CREATE OR REPLACE TABLE customer_360.gold.churn_benchmark_by_dimension AS

WITH overall AS (
  SELECT AVG(CASE WHEN is_churned THEN 1.0 ELSE 0 END) AS overall_rate
  FROM customer_360.silver.silver_churn_benchmark
)

SELECT 'Geography' AS dimension, geography AS category,
       COUNT(*) AS customers,
       ROUND(AVG(CASE WHEN is_churned THEN 1.0 ELSE 0 END) * 100, 2) AS churn_rate_pct,
       RANK() OVER (ORDER BY AVG(CASE WHEN is_churned THEN 1.0 ELSE 0 END) DESC) AS risk_rank
FROM customer_360.silver.silver_churn_benchmark
GROUP BY geography

UNION ALL

SELECT 'Gender' AS dimension, gender AS category,
       COUNT(*) AS customers,
       ROUND(AVG(CASE WHEN is_churned THEN 1.0 ELSE 0 END) * 100, 2) AS churn_rate_pct,
       RANK() OVER (ORDER BY AVG(CASE WHEN is_churned THEN 1.0 ELSE 0 END) DESC) AS risk_rank
FROM customer_360.silver.silver_churn_benchmark
GROUP BY gender

UNION ALL

SELECT 'Age Band' AS dimension, age_band AS category,
       COUNT(*) AS customers,
       ROUND(AVG(CASE WHEN is_churned THEN 1.0 ELSE 0 END) * 100, 2) AS churn_rate_pct,
       RANK() OVER (ORDER BY AVG(CASE WHEN is_churned THEN 1.0 ELSE 0 END) DESC) AS risk_rank
FROM customer_360.silver.silver_churn_benchmark
GROUP BY age_band

UNION ALL

SELECT 'Product Category' AS dimension, product_category AS category,
       COUNT(*) AS customers,
       ROUND(AVG(CASE WHEN is_churned THEN 1.0 ELSE 0 END) * 100, 2) AS churn_rate_pct,
       RANK() OVER (ORDER BY AVG(CASE WHEN is_churned THEN 1.0 ELSE 0 END) DESC) AS risk_rank
FROM customer_360.silver.silver_churn_benchmark
GROUP BY product_category

UNION ALL

SELECT 'Credit Score Band' AS dimension, credit_score_band AS category,
       COUNT(*) AS customers,
       ROUND(AVG(CASE WHEN is_churned THEN 1.0 ELSE 0 END) * 100, 2) AS churn_rate_pct,
       RANK() OVER (ORDER BY AVG(CASE WHEN is_churned THEN 1.0 ELSE 0 END) DESC) AS risk_rank
FROM customer_360.silver.silver_churn_benchmark
GROUP BY credit_score_band

UNION ALL

SELECT 'Balance Tier' AS dimension, balance_tier AS category,
       COUNT(*) AS customers,
       ROUND(AVG(CASE WHEN is_churned THEN 1.0 ELSE 0 END) * 100, 2) AS churn_rate_pct,
       RANK() OVER (ORDER BY AVG(CASE WHEN is_churned THEN 1.0 ELSE 0 END) DESC) AS risk_rank
FROM customer_360.silver.silver_churn_benchmark
GROUP BY balance_tier

UNION ALL

SELECT 'Active Status' AS dimension,
       CASE WHEN is_active THEN 'Active' ELSE 'Inactive' END AS category,
       COUNT(*) AS customers,
       ROUND(AVG(CASE WHEN is_churned THEN 1.0 ELSE 0 END) * 100, 2) AS churn_rate_pct,
       RANK() OVER (ORDER BY AVG(CASE WHEN is_churned THEN 1.0 ELSE 0 END) DESC) AS risk_rank
FROM customer_360.silver.silver_churn_benchmark
GROUP BY is_active

UNION ALL

SELECT 'Has Credit Card' AS dimension,
       CASE WHEN has_credit_card THEN 'Has Card' ELSE 'No Card' END AS category,
       COUNT(*) AS customers,
       ROUND(AVG(CASE WHEN is_churned THEN 1.0 ELSE 0 END) * 100, 2) AS churn_rate_pct,
       RANK() OVER (ORDER BY AVG(CASE WHEN is_churned THEN 1.0 ELSE 0 END) DESC) AS risk_rank
FROM customer_360.silver.silver_churn_benchmark
GROUP BY has_credit_card

UNION ALL

SELECT 'Salary Band' AS dimension, salary_band AS category,
       COUNT(*) AS customers,
       ROUND(AVG(CASE WHEN is_churned THEN 1.0 ELSE 0 END) * 100, 2) AS churn_rate_pct,
       RANK() OVER (ORDER BY AVG(CASE WHEN is_churned THEN 1.0 ELSE 0 END) DESC) AS risk_rank
FROM customer_360.silver.silver_churn_benchmark
GROUP BY salary_band;

### churn risk drivers

In [0]:
%sql

CREATE OR REPLACE TABLE customer_360.gold.churn_risk_drivers AS

WITH overall AS (
  SELECT AVG(CASE WHEN is_churned THEN 1.0 ELSE 0 END) AS overall_rate
  FROM customer_360.silver.silver_churn_benchmark
)

SELECT
  dimension,
  category,
  customers,
  churn_rate_pct,
  ROUND(churn_rate_pct - (SELECT overall_rate * 100 FROM overall), 2) AS pct_points_above_average,
  ROUND(churn_rate_pct / (SELECT overall_rate * 100 FROM overall), 2) AS times_riskier_than_average,
  RANK() OVER (ORDER BY churn_rate_pct DESC)                          AS overall_risk_rank

FROM customer_360.gold.churn_benchmark_by_dimension
ORDER BY churn_rate_pct DESC;

### churn model comparison

In [0]:
%sql

CREATE OR REPLACE TABLE customer_360.gold.churn_model_comparison AS

WITH industry AS (
  SELECT
    'Industry Benchmark' AS source,
    ROUND(AVG(CASE WHEN is_churned THEN 1.0 ELSE 0 END) * 100, 2) AS churn_rate_pct,
    COUNT(*) AS customers
  FROM customer_360.silver.silver_churn_benchmark
),

your_model AS (
  SELECT
    'Your Customer 360 Model' AS source,
    ROUND(AVG(CASE WHEN churn_risk_tier = 'High' THEN 1.0 ELSE 0 END) * 100, 2) AS churn_rate_pct,
    COUNT(*) AS customers
  FROM customer_360.gold.customer_360
)

SELECT * FROM industry
UNION ALL
SELECT * FROM your_model;

In [0]:
%sql

SELECT * FROM customer_360.gold.churn_risk_drivers ORDER BY overall_risk_rank LIMIT 10;

In [0]:
%sql
SELECT * FROM customer_360.gold.churn_model_comparison;